In [ ]:
#%pip install krippendorff

In [11]:


import make_tables as mt
mt.OUT_DIR = "runs/gpt-6-luna"   
t1 = mt.table1()
t2 = mt.table2()
t1.to_csv(f"{mt.OUT_DIR}/table1_baseline.csv")
t2.to_csv(f"{mt.OUT_DIR}/table2_conditions.csv")

/Users/lingyunfan/Desktop/semester/2026spring/501SODA/moral_project/llm_pipeline/make_tables.py:73: RuntimeWarning: Mean of empty slice
  "alpha_vs_coders_mean": np.nanmean([alpha(c, llm) for c in coders]),
/Users/lingyunfan/Desktop/semester/2026spring/501SODA/moral_project/llm_pipeline/make_tables.py:74: RuntimeWarning: Mean of empty slice
  "human_human_alpha_mean": np.nanmean(hh),


note: no baseline run 2 file found; skipping the noise-floor row


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 0 and the array at index 1 has size 50

In [9]:
import krippendorff
import pandas as pd
from dotenv import load_dotenv; load_dotenv()
import run_grid_openai as rg
from sklearn.metrics import cohen_kappa_score



def compare_runs(file_a, file_b):
    a = pd.read_csv(file_a)
    b = pd.read_csv(file_b)
    m = a.merge(b, on="para_id", suffixes=("_a", "_b"))
    rows = []
    for l in rg.LABELS:
        sub = m[[f"{l}_a", f"{l}_b"]].dropna()          # 去掉解析失败的段落
        x, y = sub[f"{l}_a"].astype(int), sub[f"{l}_b"].astype(int)
        alpha = (krippendorff.alpha(reliability_data=[x.values, y.values], level_of_measurement="nominal")
                 if len(set(x) | set(y)) > 1 else float("nan"))
        rows.append({"label": l, "n": len(sub),
                     "a_n": int(x.sum()), "b_n": int(y.sum()),
                     "pct_agree": (x == y).mean(),
                     "kripp_alpha": alpha,
                     "cohen_kappa": cohen_kappa_score(x, y)})
    return pd.DataFrame(rows).set_index("label").round(3)

# 例：baseline vs paraphrase × 0-shot
compare_runs(f"{rg.OUT_DIR}/main_0shot_validation.csv",
             f"{rg.OUT_DIR}/paraphrase_0shot_validation.csv")

,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,50,25,29,0.88,0.761,0.760
care,50,10,13,0.94,0.832,0.831
fairness,50,9,11,0.96,0.876,0.875
loyalty,50,4,3,0.98,0.848,0.847
authority,50,13,18,0.90,0.769,0.769
sanctity,50,4,5,0.98,0.879,0.878


In [7]:

for v in rg.VARIANTS:
    for k in rg.SHOTS:
        if (v, k) == rg.PRIMARY:
            continue
        print(f"main_0shot vs {v}_{k}shot")
        display(compare_runs(f"{rg.OUT_DIR}/main_0shot_validation.csv",
                             f"{rg.OUT_DIR}/{v}_{k}shot_validation.csv"))

main_0shot vs main_2shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,48,18,16,0.917,0.820,0.818
care,48,7,5,0.958,0.812,0.810
fairness,48,6,6,0.958,0.812,0.810
loyalty,48,7,7,0.833,0.338,0.331
authority,48,10,8,0.917,0.729,0.727
sanctity,48,5,5,0.958,0.779,0.777


main_0shot vs main_5shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,45,16,16,0.911,0.808,0.806
care,45,6,5,0.978,0.898,0.897
fairness,45,5,4,0.978,0.878,0.877
loyalty,45,7,8,0.889,0.604,0.600
authority,45,8,8,0.956,0.850,0.848
sanctity,45,5,5,0.956,0.778,0.775


main_0shot vs paraphrase_0shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,50,20,21,0.98,0.959,0.959
care,50,9,10,0.98,0.936,0.935
fairness,50,7,7,0.96,0.836,0.834
loyalty,50,7,6,0.86,0.387,0.382
authority,50,11,13,0.96,0.891,0.891
sanctity,50,6,6,0.96,0.812,0.811


... (6 outputs left out)

main_0shot vs reverse_2shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,48,18,21,0.854,0.701,0.699
care,48,7,10,0.938,0.788,0.787
fairness,48,6,5,0.979,0.898,0.897
loyalty,48,7,12,0.854,0.545,0.548
authority,48,10,9,0.979,0.935,0.934
sanctity,48,5,5,0.958,0.779,0.777


main_0shot vs reverse_5shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,45,16,21,0.844,0.682,0.683
care,45,6,11,0.889,0.641,0.645
fairness,45,5,6,0.933,0.693,0.690
loyalty,45,7,12,0.889,0.670,0.672
authority,45,8,9,0.978,0.928,0.928
sanctity,45,5,6,0.933,0.693,0.690
